# ShakeMap + Ground Failure: San Andreas M7.6 Demo

**BSSC 2014 scenario** — Hayward-Rodgers Creek + North San Andreas, M7.6  
Full pipeline: ShakeMap → Ground Failure → Visualization.  
Run cells top-to-bottom.

## 1. Setup

In [1]:
import os, json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

EVENT = 'bssc2014nsanandreassapsassha_m7p6_se'
os.environ['EVENT'] = EVENT

PRODUCTS = Path.home() / 'shakemap_profiles/default/data' / EVENT / 'current/products'
GF_OUT   = Path.home() / 'gf_output' / EVENT
CA_DATA  = Path('/workspaces/shakemap-codespaces/data/california_inputs')

print(f'Event:    {EVENT}')
print(f'Products: {PRODUCTS}')
print(f'GF data:  {CA_DATA}')

Event:    bssc2014nsanandreassapsassha_m7p6_se
Products: /home/vscode/shakemap_profiles/default/data/bssc2014nsanandreassapsassha_m7p6_se/current/products
GF data:  /workspaces/shakemap-codespaces/data/california_inputs


In [5]:
%%bash
sm_create bssc2014nsanandreassapsassha_m7p6_se

Wrote 2 files to /home/vscode/shakemap_profiles/default/data/bssc2014nsanandreassapsassha_m7p6_se/current:
	rupture.json
	event.xml


## 2. ShakeMap inputs

The event was pre-staged with `sm_create`. Key input files:
- **`event.xml`** — origin: location, depth, magnitude, time
- **`rupture.json`** — finite fault geometry

In [6]:
event_dir = Path.home() / 'shakemap_profiles/default/data' / EVENT / 'current'
print('=== event.xml ===')
print((event_dir / 'event.xml').read_text())

=== event.xml ===
<earthquake id="bssc2014nsanandreassapsassha_m7p6_se" netid="bssc2014" network="bssc2014" lat="37.2972" lon="-122.1401" depth="7.8" mag="7.6" time="2017-06-30T16:03:44Z" locstring="N. San Andreas: SAP+SAS" mech="ALL" event_type="SCENARIO" reviewed="unknown"/>



In [7]:
rup = json.loads((event_dir / 'rupture.json').read_text())
print('Type:', rup.get('type'))
print('Features:', len(rup.get('features', [])))
print('Reference:', rup.get('metadata', {}).get('reference', 'n/a'))

Type: FeatureCollection
Features: 1
Reference: 


## 3. Run ShakeMap

Four modules run in sequence:
- **`assemble`** — collect all inputs
- **`model`** — compute ground motion using GMPEs + Vs30 site corrections (~3 min)
- **`contour`** — shaking contour lines
- **`mapping`** — render map images

In [8]:
%%bash
shake $EVENT assemble -c 'demo' model contour mapping 2>&1 \
  | grep -E 'Running|Finished|ERROR|WARNING'

INFO -- 2026-07-27 20:38:02 -- shake.main -- Running command assemble
INFO -- 2026-07-27 20:38:02 -- shake.main -- Finished running command assemble: Elapsed 0.18 secs
INFO -- 2026-07-27 20:38:02 -- shake.main -- Running command model
INFO -- 2026-07-27 20:41:26 -- shake.main -- Finished running command model: Elapsed 204.42 secs
INFO -- 2026-07-27 20:41:26 -- shake.main -- Running command contour
INFO -- 2026-07-27 20:41:31 -- shake.main -- Finished running command contour: Elapsed 5.14 secs
INFO -- 2026-07-27 20:41:32 -- shake.main -- Running command mapping
INFO -- 2026-07-27 20:42:32 -- shake.main -- Finished running command mapping: Elapsed 60.14 secs


In [9]:
cat ~/shakemap_profiles/default/data/bssc2014nsanandreassapsassha_m7p6_se/current/products/grid.xml | head -20

cat: /home/vscode/shakemap_profiles/default/data/bssc2014nsanandreassapsassha_m7p6_se/current/products/grid.xml: No such file or directory


In [ ]:
%%bash
shake $EVENT gridxml 2>&1 | grep -E 'Running|Finished|ERROR'

In [ ]:
print('Output products:')
for f in sorted(PRODUCTS.glob('*')):
    print(f'  {f.name:45s} {f.stat().st_size/1e6:.1f} MB')

## 4. Visualize ShakeMap outputs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, name, title in [
    (axes[0], 'intensity.jpg', 'MMI Intensity'),
    (axes[1], 'pga.jpg',       'PGA'),
]:
    img_path = PRODUCTS / name
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
    ax.set_title(title, fontsize=13)
    ax.axis('off')
plt.suptitle('San Andreas M7.6 — ShakeMap', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Run ground failure

Ground failure reads the ShakeMap `grid.xml` and estimates:
- **Landslide probability** — Nowicki Jessee et al. (2018)
- **Liquefaction probability** — Zhu et al. (2017)

The `gf` conda environment is used via its absolute path.

In [ ]:
GRID = PRODUCTS / 'grid.xml'
print(f'Grid exists: {GRID.exists()}')
print(f'CA data exists: {CA_DATA.exists()}')

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d /workspaces/shakemap-codespaces/data/california_inputs \
  2>&1 | tail -5

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d /workspaces/shakemap-codespaces/data/california_inputs \
  2>&1 | tail -5

In [ ]:
print('Ground failure outputs:')
for f in sorted(GF_OUT.glob('*.tif')):
    print(f'  {f.name}')

## 6. Static 3-panel ground failure map

In [ ]:
%%bash
python ~/plot_gf.py \
  --shakefile ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --datadir /workspaces/shakemap-codespaces/data/california_inputs \
  --outfile ~/san_andreas_gf.png 2>&1 | tail -3

In [ ]:
img = mpimg.imread(str(Path.home() / 'san_andreas_gf.png'))
fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Interactive two-panel map

Landslide (left) and liquefaction (right) side-by-side with shaking contours.
Click anywhere to see coordinates. Toggle layers in the bottom-right.

In [ ]:
%%bash
/opt/conda/envs/gf/bin/python ~/plot_gf_interactive.py \
  --ls-model "Jessee 2018:$HOME/gf_output/$EVENT/${EVENT}_jessee_2018_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini" \
  --lq-model "Zhu 2017:$HOME/gf_output/$EVENT/${EVENT}_zhu_2017_general_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini" \
  --shakefile ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --contours  ~/shakemap_profiles/default/data/$EVENT/current/products/cont_mmi.json \
  --outfile ~/san_andreas_gf.html 2>&1 | tail -5

In [ ]:
from IPython.display import IFrame
IFrame(src=str(Path.home() / 'san_andreas_gf.html'), width='100%', height=600)

---
**End of demo.**  
Download `san_andreas_gf.html` from the Explorer (right-click → Download) to open locally in a browser.